In [1]:
# Cell 1 — imports & paths
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import yaml
from core.orchestrator import load_yaml, encode_vector, decode_vector
from core.optimizer.pso import PSO
from core.database import RunDB
from core.cache import ParamCache
from models.hss import HSS

CFG = Path("../config")

In [2]:
# Cell 2 — parse configs, build bounds
project = load_yaml(CFG/"project.yaml")
pso_cfg = load_yaml(CFG/"pso.yaml")
profile = load_yaml(CFG/"soil_profile.yaml")
targets = load_yaml(CFG/"targets.yaml")
bounds, syms = encode_vector(profile, profile["bounds_mode"])
print(f"Search dimensionality: {len(syms)}")
for (layer, p, scale), b in zip(syms, bounds):
    print(f"  {layer:>12s}.{p:<10s}  scale={scale:<6s}  bounds=({b[0]:.3f}, {b[1]:.3f})")

Search dimensionality: 12
    Sand_Upper.E50_ref     scale=log     bounds=(10.309, 11.290)
    Sand_Upper.Eoed_ref    scale=log     bounds=(10.309, 11.290)
    Sand_Upper.Eur_ref     scale=log     bounds=(11.408, 12.388)
    Sand_Upper.G0_ref      scale=log     bounds=(11.918, 12.899)
    Sand_Upper.m           scale=linear  bounds=(0.500, 0.700)
    Sand_Upper.phi         scale=linear  bounds=(32.000, 40.000)
    Sand_Upper.c_ref       scale=linear  bounds=(0.000, 5.000)
    Sand_Upper.gamma_07    scale=log     bounds=(-8.805, -8.112)
     Weak_Rock.Eur_ref     scale=log     bounds=(12.899, 13.998)
     Weak_Rock.G0_ref      scale=log     bounds=(13.592, 14.509)
     Weak_Rock.phi         scale=linear  bounds=(38.000, 45.000)
     Weak_Rock.c_ref       scale=linear  bounds=(100.000, 300.000)


In [3]:
# Cell 3 — analytical toy fitness (no PLAXIS) to prove the loop
# Pretend the "true" parameters live at midpoint of each range.
true_x = bounds.mean(axis=1)
def toy_fitness(x):
    return float(np.sum((x - true_x)**2 / (bounds[:,1]-bounds[:,0])**2))

In [4]:
# Cell 4 — run PSO end-to-end
rng = np.random.default_rng(project["reproducibility"]["seed"])
opt = PSO(n_particles=pso_cfg["swarm"]["size"], n_dims=len(syms),
          bounds=bounds, config=pso_cfg, rng=rng)
for it in range(pso_cfg["iterations"]["max"]):
    gb, gf, fits = opt.step(toy_fitness)
    if (it % 5) == 0:
        print(f"iter {it:>3d}  gbest={gf:.4e}  mean={fits.mean():.4e}")
    if opt.should_stop():
        print("Early stop."); break
print("\nFinal gbest decoded:")
print(yaml.safe_dump(decode_vector(gb, syms, profile), sort_keys=False))

iter   0  gbest=6.5063e-01  mean=9.8286e-01
iter   5  gbest=2.4105e-01  mean=6.6706e-01
iter  10  gbest=7.9936e-02  mean=7.0693e-01
iter  15  gbest=7.9936e-02  mean=4.8130e-01
Early stop.

Final gbest decoded:
Sand_Upper:
  psi: 0.0
  nu_ur: 0.2
  Rf: 0.9
  pref: 100.0
  E50_ref: 49326.87026908866
  Eoed_ref: 49891.67774823348
  Eur_ref: 153376.92044530896
  G0_ref: 220922.3711045328
  m: 0.5992876755484934
  phi: 36.40764654426854
  c_ref: 3.3869302749775247
  gamma_07: 0.00021678828745480062
Weak_Rock:
  psi: 0.0
  nu_ur: 0.2
  Rf: 0.9
  pref: 100.0
  m: 0.5
  gamma_07: 0.0002
  E50_ref: 400.0e3
  Eoed_ref: 400.0e3
  Eur_ref: 594321.9354042877
  G0_ref: 1228178.098067391
  phi: 42.11675079210158
  c_ref: 211.37023055629896



In [5]:
# Cell 5 — verify DB I/O
db = RunDB("./_smoke_test.db")
rid = db.insert_run(0, 0, "ok", 0.123, 0.045, "smoke")
db.insert_parameters(rid, decode_vector(gb, syms, profile))
db.insert_metrics(rid, [("nrmse","total",0.045),("rmse","upper",0.041)])
print(db.to_dataframe("runs"))
print(db.to_dataframe("parameters").head())

   run_id  particle_id  iteration status  walltime_s  fitness  \
0       1            0          0     ok       0.123    0.045   

             timestamp  notes  
0  2026-06-20 21:02:41  smoke  
   run_id       layer param_name         value
0       1  Sand_Upper        psi      0.000000
1       1  Sand_Upper      nu_ur      0.200000
2       1  Sand_Upper         Rf      0.900000
3       1  Sand_Upper       pref    100.000000
4       1  Sand_Upper    E50_ref  49326.870269


In [6]:
# Cell 6 — exercise the HSS constraint checker
hss = HSS()
bad = dict(E50_ref=50e3, Eur_ref=100e3, Eoed_ref=50e3, G0_ref=10e3, nu_ur=0.2)
print("violations:", hss.physical_constraints(bad))
good = dict(E50_ref=50e3, Eur_ref=180e3, Eoed_ref=50e3, G0_ref=200e3, nu_ur=0.2,
            phi=35, c_ref=0, gamma_07=2e-4)
print("violations:", hss.physical_constraints(good))

violations: ['Eur_ref (100000) < 3*E50_ref (150000)', 'G0_ref < Eur_ref/(2(1+nu_ur)) = 41667']
violations: []


In [7]:
# Cell 7 — exercise cache
cache = ParamCache(1e-6)
cache.put(gb, 0.045)
print("cached fitness:", cache.get(gb))

cached fitness: 0.045
